# Week 08 — Home exercise 2: Summarizing the panel

**Solution proposal.**

Aggregating, transforming, and the baseline pattern that turns a level into an anomaly.

In [1]:
import numpy as np
import pandas as pd

co2 = pd.read_csv("../data/co2_emissions.csv")
co2 = co2.dropna(subset=["co2_total"])
co2["co2_pc"] = co2["co2_total"] * 1_000_000 / co2["population"]

print(co2.shape)

(5904, 12)


## 1. A summary by year

In [2]:
by_year = co2.groupby("year").agg(
    entities=("co2_total", "count"),
    total_co2=("co2_total", "sum"),
    mean_pc=("co2_pc", "mean"),
).round(2)

by_year.tail(5)

,entities,total_co2,mean_pc
year,,,
2019,246,305111.32,4.62
2020,246,295477.90,4.35
2021,246,312230.85,4.53
2022,246,314199.08,4.50
2023,246,320155.60,4.41


246 entities in every year — a balanced panel, as we knew. Emissions per person drift slowly
downward while the total rises, which is population growth outrunning per-person improvement.

Note that `entities` uses `count` on `co2_total` rather than `size`: we want how many entities have a
figure, not how many rows there are. Here the two agree because we dropped the rows with no figure
when we loaded the file — but they would not have before that, and asking the right one is free.

## 2. By income group, in 2023

The bands come from last week. The `.loc` line afterwards is the one that matters: without it, every
row with no `gdp_pc` fails all three conditions, falls through to `default`, and is labeled high
income.

In [3]:
conditions = [
    co2["gdp_pc"] < 1_000,
    co2["gdp_pc"] < 10_000,
    co2["gdp_pc"] < 50_000,
]

co2["income_group"] = np.select(conditions, ["low", "lower-middle", "upper-middle"], default="high")
co2.loc[co2["gdp_pc"].isna(), "income_group"] = None

year_2023 = co2[co2["year"] == 2023]

by_income = year_2023.groupby("income_group").agg(
    entities=("co2_total", "count"),
    total_co2=("co2_total", "sum"),
    mean_pc=("co2_pc", "mean"),
).round(2)

by_income

,entities,total_co2,mean_pc
income_group,,,
high,26,22048.12,8.96
low,17,78.02,0.19
lower-middle,116,141982.29,2.06
upper-middle,76,155918.02,7.68


### Do the groups add up?

In [4]:
print("sum of the groups:", by_income["total_co2"].sum().round(1))
print("sum of the column:", year_2023["co2_total"].sum().round(1))
print("difference:       ", (year_2023["co2_total"].sum() - by_income["total_co2"].sum()).round(1))
print()
print("rows with no income_group:", year_2023["income_group"].isna().sum())

sum of the groups: 320026.4
sum of the column: 320155.6
difference:        129.2

rows with no income_group: 11


**No, they do not, and 129.2 million tonnes are missing.**

Eleven entities in 2023 have no `gdp_pc`, so they have no `income_group`, so `groupby` has no group to
put them in and drops them. The four group totals are each correct, and their sum is not the total.

This is the same fault as `dropna()` deleting rows you were not thinking about, moved one step later
in the analysis. The check that catches it is one line, and it is the line to write after every
`groupby` on a key that could be missing.

## 3. Each row against its own history

The group's average has to sit beside every row of that group, so this is `transform`, not an
aggregation.

In [5]:
co2["own_mean_pc"] = co2.groupby("country")["co2_pc"].transform("mean")
co2["pct_of_own_mean"] = 100 * co2["co2_pc"] / co2["own_mean_pc"]

co2[co2["country"] == "Norway"][["year", "co2_pc", "own_mean_pc", "pct_of_own_mean"]].tail(5).round(2)

,year,co2_pc,own_mean_pc,pct_of_own_mean
4291,2019,8.48,9.09,93.26
4292,2020,8.12,9.09,89.36
4293,2021,8.21,9.09,90.35
4294,2022,7.98,9.09,87.76
4295,2023,7.54,9.09,82.90


Norway in 2023 emitted 82.9% of its own 24-year average per person — the lowest of the five, and part
of a run of falls since 2019.

## A baseline over part of the period

The baseline is an average over **2000–2009 only**, and it has to end up beside every row from 2000 to
2023. `transform` cannot do that: it computes over the rows in the group it is given, and the group
here is all 24 years.

So the baseline becomes its own table, and comes back with a merge.

In [6]:
baseline_years = (co2["year"] >= 2000) & (co2["year"] <= 2009)

baseline = (
    co2[baseline_years]
    .groupby("country")["co2_pc"]
    .mean()
    .reset_index()
    .rename(columns={"co2_pc": "baseline_pc"})
)

print(baseline.shape)
baseline.head(3).round(3)

(246, 2)


,country,baseline_pc
0,Afghanistan,0.078
1,Africa Eastern and Southern,1.102
2,Africa Western and Central,0.489


`reset_index()` turns the grouped Series into a two-column table, and `rename` stops the merge from
producing `co2_pc_x` and `co2_pc_y`. One row per entity, so the merge is many-to-one — and we say so.

In [7]:
before = len(co2)

co2 = co2.merge(baseline, on="country", how="left", validate="many_to_one")
co2["anomaly"] = co2["co2_pc"] - co2["baseline_pc"]

print("rows before:", before)
print("rows after: ", len(co2))

rows before: 5904
rows after:  5904


Row count unchanged, which is what a many-to-one left join should do and what we would want to know
about if it had not.

In [8]:
top_2023 = (
    co2[co2["year"] == 2023]
    .sort_values("anomaly", ascending=False)
    [["country", "co2_pc", "baseline_pc", "anomaly"]]
)

top_2023.head(5).round(2)

,country,co2_pc,baseline_pc,anomaly
3911,New Caledonia,16.78,10.59,6.20
1079,China,9.23,4.49,4.74
4151,Oman,18.96,14.62,4.34
2135,Greenland,10.29,6.43,3.85
1487,East Asia & Pacific (IDA & IBRD countries),7.11,3.63,3.48


### The one that is not a country

**`East Asia & Pacific (IDA & IBRD countries)`** is a World Bank grouping, not a country, and it is
sitting fifth in a list of countries.

Three things give it away, in increasing order of reliability:

1. **The name.** A region and a parenthesis full of acronyms is not a country. This works here and
   will not work in general — plenty of real countries have long names, and plenty of aggregates have
   short ones.
2. **The population.** It is far larger than any of its neighbors in the table, because it contains
   several hundred million people from many countries.
3. **A second table that says so.** `country_info.csv` labels every entity with a region, and marks
   the groupings `"Aggregates"`. That is the only method that scales, and it is the one to use.

Without it the whole list is suspect: New Caledonia and Greenland are real, but nothing in *this*
file distinguishes them from the aggregate three rows below.

### Things worth noticing

- **`transform` and `merge` solve the same problem in different circumstances.** Use `transform` when
  the statistic is computed over the same rows it has to sit beside. Use a small table plus a merge
  when it is computed over a subset — a baseline period, a reference year, a different dataset.
- **`reset_index()` then `rename()` is the standard preparation** for bringing a grouped result back
  as a column. Skipping the rename gives you `_x` and `_y` suffixes and a bad afternoon.
- The `anomaly` column is `NaN` for any entity with no `co2_pc` in 2000–2009, which is correct: there
  is no baseline to measure against. `sort_values` puts those at the end rather than pretending they
  are zero.

### What this notebook does NOT do

- It never removes the aggregates, so every ranking in it is contaminated. That needs the other file.
- The baseline is an unweighted mean of whatever years happen to be present between 2000 and 2009. An
  entity observed in only two of those ten years gets a baseline anyway, and it is a much noisier one
  — nothing here reports how many years each baseline was built from, and it should.
- `pct_of_own_mean` divides by a mean that includes the row itself. Over 24 years that hardly matters;
  over four it would.